# SARC balanced Reddit corpus — cleaning pipeline

This notebook combines `text_cleaning.py` (normalisation, label-leak removal,
feature extraction), `report.py` (the `CleaningLog` audit log), and
`01_clean_primary.py` (the cleaning pipeline that uses both) into **one
self-contained notebook**. Both `text_cleaning` and `report` are inlined in
full below, so this notebook does **not** need `common/text_cleaning.py` or
`common/report.py` on `sys.path` — every cell can run top-to-bottom on its
own.

> **Still required:** this notebook still imports `common.paths`
> (`CLEAN_REPORTS`, `PROCESSED`, `RAW_PRIMARY`, `SAMPLES`, `ensure_dirs`),
> which wasn't provided. Keep a `common/paths.py` module next to this
> notebook (defining those four `Path` constants plus an `ensure_dirs()`
> helper), or share its source and I'll inline it too for zero external
> file dependencies.

Pipeline:
1. load + type-coerce
2. drop rows with an unusable comment (`[deleted]`/`[removed]`/empty/NaN)
3. normalise unicode, strip Reddit markdown, replace URLs/mentions
4. remove label leakage (`/s`, `\s`, `"#sarcasm"`, `<sarcasm>`) — critical
5. drop too-short / absurdly long comments
6. English filter (lexical heuristic + optional langdetect adjudication)
7. de-duplicate: exact, then near-duplicate on (text, label)
8. engineer surface + context + temporal features
9. write parquet + a stratified CSV sample + an audit report


In [1]:
from __future__ import annotations

import html
import json
import re
import sys
import time
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "scripts"))  # for common.paths — adjust if needed
#from common.paths import CLEAN_REPORTS, PROCESSED, RAW_PRIMARY, SAMPLES, ensure_dirs

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

## Part 1 — `text_cleaning`: shared normalisation, label-leak removal, features

Design notes:
* The order of operations matters. We count surface features (caps, "!!!",
  emoji, elongation) BEFORE lowercasing/normalising, because those very
  features are the sarcasm markers the project is trying to explain.
* Label leakage is treated as a first-class concern. Reddit labels come from
  the author writing "/s"; Twitter labels come from "#sarcasm". If those
  tokens survive into the text, a classifier scores ~100% by memorising them
  and the whole explainability story collapses. `strip_label_markers` removes
  them and reports how often they occurred.
* Language filtering is two-stage: a cheap lexical heuristic decides the clear
  cases, and (optionally) `langdetect` adjudicates only the ambiguous band.
  A full langdetect pass over 1M short comments takes ~20 min and is
  unreliable on very short strings; this keeps it to a few percent of rows.

### Regexes (compiled once; these run over ~1M rows)

In [2]:
RE_URL = re.compile(r"https?://\S+|www\.\S+", re.I)
RE_MD_LINK = re.compile(r"\[([^\]]*)\]\(\s*<?https?://[^)\s]+>?\s*\)")
RE_REDDIT_USER = re.compile(r"(?<![\w/])/?u/[A-Za-z0-9_\-]+", re.I)
RE_REDDIT_SUB = re.compile(r"(?<![\w/])/?r/([A-Za-z0-9_]+)", re.I)
RE_TW_MENTION = re.compile(r"(?<![\w@])@[A-Za-z0-9_]{1,15}\b")
RE_HASHTAG = re.compile(r"#(\w+)")
RE_CODEBLOCK = re.compile(r"(?:^ {4}.*$\n?)+|`{1,3}[^`]*`{1,3}", re.M)
# Quote lines are matched AFTER html unescaping, so the marker is a bare ">".
# The escaped forms are kept as alternatives because some rows are triple-
# escaped and one pass of unescaping leaves "&gt;" behind.
RE_QUOTELINE = re.compile(r"^[ \t]*(?:&(?:amp;)*gt;|>)+.*$", re.M)
RE_MD_EMPH = re.compile(r"(\*{1,3}|_{1,3}|~{2})(?=\S)(.+?)(?<=\S)\1", re.S)
RE_SUPER = re.compile(r"\^+")
RE_ENTITY_NUM = re.compile(r"&#x?[0-9a-fA-F]+;")
RE_ZERO_WIDTH = re.compile(r"[​-‏‪-‮﻿­]")
RE_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
RE_WS = re.compile(r"\s+")

# Label-leak markers -------------------------------------------------------
# Reddit: a trailing "/s", "\s", "(/s)", "[/s]", "-/s", optionally after
# punctuation. We deliberately require it to stand alone as a token so we do
# not destroy legitimate text like "w/s" or a file path.
RE_SLASH_S = re.compile(
    r"""(?:^|(?<=[\s.,!?;:"'\)\]\}]))      # start or after a boundary char
        [\(\[\{\-~]*                        # optional opening decoration
        [/\\]\s?s                           # the marker itself: /s or \s
        [\)\]\}\.\!]*                       # optional closing decoration
        (?=$|[\s.,!?;:"'\(\[\{])            # must end at a boundary
    """,
    re.I | re.X,
)
LABEL_HASHTAGS = {
    "sarcasm", "sarcastic", "sarcastictweet", "sarcasmtweet", "notsarcasm",
    "irony", "ironic", "ironia", "not", "justkidding", "jk", "kidding",
    "sarcastically", "sarcasam",
}

# Surface-marker regexes ---------------------------------------------------
RE_EMOJI = re.compile(
    "[" "\U0001f300-\U0001faff" "\U00002600-\U000027bf"
    "\U0001f1e6-\U0001f1ff" "\U00002190-\U000021ff" "\U00002b00-\U00002bff"
    "\U0000fe0f" "]"
)
RE_ELONG = re.compile(r"([A-Za-z])\1{2,}")
RE_REPEAT_PUNCT = re.compile(r"([!?.])\1{1,}")
RE_ALLCAPS = re.compile(r"\b[A-Z]{2,}\b")
RE_SCARE_QUOTE = re.compile(r"[\"“”'‘’][^\"“”]{1,40}[\"“”]")
RE_WORD = re.compile(r"[A-Za-z']+")

DELETED_MARKERS = {"[deleted]", "[removed]", "deleted", "removed", "nan", "none", ""}

# Interjections / stance markers repeatedly flagged in the sarcasm literature
INTERJECTIONS = {
    "oh", "ah", "wow", "yeah", "yea", "yep", "sure", "right", "great", "nice",
    "gee", "gosh", "huh", "hmm", "well", "ok", "okay", "lol", "haha", "obviously",
    "clearly", "totally", "absolutely", "definitely", "shocking", "shocker",
    "wonderful", "brilliant", "genius", "amazing", "perfect", "fantastic",
}

# High-frequency English function words for the cheap language heuristic.
EN_FUNCTION_WORDS = {
    "the", "be", "to", "of", "and", "a", "in", "that", "have", "i", "it", "for",
    "not", "on", "with", "he", "as", "you", "do", "at", "this", "but", "his",
    "by", "from", "they", "we", "say", "her", "she", "or", "an", "will", "my",
    "one", "all", "would", "there", "their", "what", "so", "up", "out", "if",
    "about", "who", "get", "which", "go", "me", "when", "make", "can", "like",
    "time", "no", "just", "him", "know", "take", "people", "into", "year",
    "your", "good", "some", "could", "them", "see", "other", "than", "then",
    "now", "look", "only", "come", "its", "over", "think", "also", "back",
    "after", "use", "two", "how", "our", "work", "first", "well", "way", "even",
    "new", "want", "because", "any", "these", "give", "day", "most", "us",
    "is", "are", "was", "were", "been", "has", "had", "did", "does", "am",
    "too", "very", "really", "still", "never", "always", "here", "thing",
}

### Step 1: unicode / markup normalisation

In [3]:
def fix_mojibake(text: str) -> str:
    """
    Repair the common UTF-8-read-as-cp1252 damage (â€™ -> ').

    cp1252 is tried first because it is what actually caused the damage;
    latin-1 is the fallback for strings containing bytes cp1252 leaves
    undefined (0x81, 0x8d, 0x8f, 0x90, 0x9d), which are common in this data.
    Both round-trips are strict: if the result is not valid UTF-8 the string
    was not mojibake and is returned untouched.
    """
    if not ("Ã" in text or "â€" in text or "Â" in text):
        return text
    for codec in ("cp1252", "latin-1"):
        try:
            return text.encode(codec, errors="strict").decode("utf-8", errors="strict")
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
    return text


def normalise_unicode(text: str) -> str:
    text = fix_mojibake(text)
    # Reddit dumps are frequently double-escaped: &amp;gt; -> &gt; -> >
    for _ in range(2):
        if "&" in text:
            text = html.unescape(text)
    text = RE_ZERO_WIDTH.sub("", text)
    text = RE_CTRL.sub(" ", text)
    text = unicodedata.normalize("NFKC", text)
    # Normalise the many unicode quote/dash variants so "scare quote"
    # detection and tokenisation stay consistent across corpora.
    text = (text.replace("‘", "'").replace("’", "'")
                .replace("“", '"').replace("”", '"')
                .replace("–", "-").replace("—", "-")
                .replace("…", "..."))
    return text


def strip_markup(text: str, keep_subreddit: bool = True) -> str:
    """Remove Reddit markdown / quoting, keeping the human-readable content."""
    text = RE_CODEBLOCK.sub(" ", text)
    text = RE_QUOTELINE.sub(" ", text)          # quoted parent text, not the author's
    text = RE_MD_LINK.sub(r"\1", text)          # [anchor](url) -> anchor
    text = RE_MD_EMPH.sub(r"\2", text)          # **bold** -> bold
    text = RE_SUPER.sub("", text)
    text = RE_ENTITY_NUM.sub(" ", text)
    return text


def replace_entities(text: str, keep_subreddit: bool = True) -> str:
    """Swap volatile identifiers for stable placeholders."""
    text = RE_URL.sub(" <URL> ", text)
    text = RE_REDDIT_USER.sub(" <USER> ", text)
    text = RE_TW_MENTION.sub(" <USER> ", text)
    if keep_subreddit:
        # A subreddit name is topical signal for sub-question 3; keep the word.
        text = RE_REDDIT_SUB.sub(r" \1 ", text)
    else:
        text = RE_REDDIT_SUB.sub(" <SUB> ", text)
    return text

### Step 2: label-leak removal

In [4]:
@dataclass
class LeakCounts:
    slash_s: int = 0
    label_hashtag: int = 0
    explicit_word: int = 0


def strip_label_markers(text: str) -> tuple[str, bool, bool, bool]:
    """
    Remove the annotation artefacts that *are* the label.

    Returns (clean_text, had_slash_s, had_label_hashtag, had_explicit_word).
    """
    had_slash_s = bool(RE_SLASH_S.search(text))
    if had_slash_s:
        text = RE_SLASH_S.sub(" ", text)

    had_tag = False

    def _hash(m: re.Match) -> str:
        nonlocal had_tag
        tag = m.group(1)
        if tag.lower() in LABEL_HASHTAGS:
            had_tag = True
            return " "
        return " " + tag + " "          # keep the word, drop the '#'

    text = RE_HASHTAG.sub(_hash, text)

    # "/sarcasm", "<sarcasm>", "(sarcasm)" written out in full
    had_word = bool(re.search(r"[/<(\[]\s*sarcas(m|tic)\s*[>)\]]?", text, re.I))
    if had_word:
        text = re.sub(r"[/<(\[]\s*sarcas(m|tic)\s*[>)\]]?", " ", text, flags=re.I)

    return text, had_slash_s, had_tag, had_word

### Step 3: surface-marker features (computed on the pre-lowercased text)

In [5]:
def surface_features(text: str) -> dict:
    words = RE_WORD.findall(text)
    n_words = len(words)
    caps = RE_ALLCAPS.findall(text)
    lower = [w.lower() for w in words]
    return {
        "n_chars": len(text),
        "n_words": n_words,
        "n_sentences": max(1, len(re.findall(r"[.!?]+", text))),
        "avg_word_len": float(np.mean([len(w) for w in words])) if words else 0.0,
        "n_exclam": text.count("!"),
        "n_question": text.count("?"),
        "n_ellipsis": len(re.findall(r"\.{2,}", text)),
        "n_repeat_punct": len(RE_REPEAT_PUNCT.findall(text)),
        "n_allcaps_words": len(caps),
        "allcaps_ratio": len(caps) / n_words if n_words else 0.0,
        "n_emoji": len(RE_EMOJI.findall(text)),
        "n_elongation": len(RE_ELONG.findall(text)),
        "n_scare_quotes": len(RE_SCARE_QUOTE.findall(text)),
        "n_urls": len(RE_URL.findall(text)),
        "n_mentions": len(RE_TW_MENTION.findall(text)) + len(RE_REDDIT_USER.findall(text)),
        "n_hashtags": len(RE_HASHTAG.findall(text)),
        "n_interjections": sum(1 for w in lower if w in INTERJECTIONS),
        "starts_with_interjection": bool(lower and lower[0] in INTERJECTIONS),
    }

### Step 4: language heuristic

In [6]:
def english_score(text: str) -> float:
    """
    Cheap English-likeness score in [0, 1].

    Combines (a) the share of tokens that are English function words with
    (b) the share of letters that are ASCII. Short texts get a prior nudge so
    that a valid three-word comment is not thrown away for lacking stopwords.
    """
    words = RE_WORD.findall(text.lower())
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    ascii_ratio = sum(1 for c in letters if c.isascii()) / len(letters)
    if not words:
        return 0.35 * ascii_ratio
    fw = sum(1 for w in words if w in EN_FUNCTION_WORDS) / len(words)
    # With <=5 tokens, absence of a function word is uninformative.
    prior = 0.30 if len(words) <= 5 else 0.0
    return min(1.0, 0.65 * ascii_ratio + 0.35 * min(1.0, fw * 2.5 + prior))


def add_language_flags(df: pd.DataFrame, col: str,
                       low: float = 0.55, high: float = 0.72,
                       use_langdetect: bool = True) -> pd.DataFrame:
    """
    Two-stage language decision.

    score >= high  -> English
    score <  low   -> not English
    in between     -> ask langdetect (if installed); otherwise accept.
    """
    df["en_score"] = df[col].map(english_score)
    df["is_english"] = df["en_score"] >= high
    ambiguous = (df["en_score"] >= low) & (df["en_score"] < high)
    df["lang_method"] = np.where(df["en_score"] >= high, "heuristic-high",
                         np.where(df["en_score"] < low, "heuristic-low", "ambiguous"))

    if ambiguous.any() and use_langdetect:
        try:
            from langdetect import DetectorFactory, detect
            DetectorFactory.seed = 0

            def _d(t: str) -> bool:
                try:
                    return detect(t) == "en"
                except Exception:       # noqa: BLE001  (langdetect raises on short/empty)
                    return True         # keep it; the heuristic already liked it enough
            df.loc[ambiguous, "is_english"] = df.loc[ambiguous, col].map(_d)
            df.loc[ambiguous, "lang_method"] = "langdetect"
        except ImportError:
            df.loc[ambiguous, "is_english"] = True
            df.loc[ambiguous, "lang_method"] = "ambiguous-kept"
    elif ambiguous.any():
        df.loc[ambiguous, "is_english"] = True
        df.loc[ambiguous, "lang_method"] = "ambiguous-kept"
    return df

### Step 5: the full pipeline for one text column

In [7]:
def squash_elongation(text: str) -> str:
    """
    "sooooo" -> "soo". Off by default in `clean_series`.

    Trade-off: leaving elongation intact preserves the marker verbatim but
    fragments the vocabulary ("sooo", "soooo", "sooooo" are three types for a
    bag-of-words model). Squashing to two characters keeps the distinction
    from the ordinary word while collapsing the variants — and `n_elongation`
    has already recorded that it happened, so no signal is lost.
    """
    return RE_ELONG.sub(r"\1\1", text)


def clean_series(s: pd.Series, keep_subreddit: bool = True,
                 squash_elong: bool = False) -> pd.DataFrame:
    """
    Run the whole text pipeline over a Series.

    Returns a DataFrame with the cleaned text, the leak flags and all the
    surface features, indexed like the input.

    `squash_elong=True` additionally collapses character elongation in
    `text_clean` (the feature count is taken first, so nothing is lost).
    """
    s = s.fillna("").astype(str)

    norm = s.map(normalise_unicode)
    norm = norm.map(lambda t: strip_markup(t))

    leaks = norm.map(strip_label_markers)
    text = leaks.map(lambda x: x[0])
    out = pd.DataFrame({
        "had_slash_s": leaks.map(lambda x: x[1]),
        "had_label_hashtag": leaks.map(lambda x: x[2]),
        "had_explicit_sarcasm_word": leaks.map(lambda x: x[3]),
    }, index=s.index)

    text = text.map(lambda t: replace_entities(t, keep_subreddit))

    # Features come from the entity-replaced but still case-preserving text.
    feats = pd.DataFrame(list(text.map(surface_features)), index=s.index)

    if squash_elong:
        text = text.map(squash_elongation)
    text = text.map(lambda t: RE_WS.sub(" ", t).strip())
    out["text_clean"] = text
    out["text_norm"] = text.str.lower()            # for dedup / bag-of-words
    return pd.concat([out, feats], axis=1)


def dedup_key(s: pd.Series) -> pd.Series:
    """Aggressive normalisation used only for near-duplicate detection."""
    return (s.str.lower()
             .str.replace(r"[^a-z0-9 ]", "", regex=True)
             .str.replace(r"\s+", " ", regex=True)
             .str.strip())


def is_deleted(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.strip().str.lower().isin(DELETED_MARKERS)


def jaccard(a: str, b: str) -> float:
    """Token overlap between a comment and its parent — a cheap proxy for
    'is the reply echoing the parent', one of the classic sarcasm cues."""
    ta = set(RE_WORD.findall(a.lower()))
    tb = set(RE_WORD.findall(b.lower()))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)

## Part 2 — `report`: the `CleaningLog` audit log

A tiny audit log so every dropped row is accounted for. Records, per step,
how many rows were removed and why, plus arbitrary notes (leak rates,
language-method counts, final label balance, etc.), and can print a summary
table or save the whole thing as JSON.

In [8]:
"""A tiny audit log so every dropped row is accounted for."""


class CleaningLog:
    def __init__(self, dataset: str, n_start: int):
        self.dataset = dataset
        self.n_start = n_start
        self.n_current = n_start
        self.steps: list[dict] = []
        self.stats: dict = {}
        self.t0 = time.time()

    def drop(self, reason: str, n_after: int, detail: str = "") -> None:
        removed = self.n_current - n_after
        self.steps.append({
            "step": len(self.steps) + 1,
            "reason": reason,
            "removed": removed,
            "pct_of_original": round(100 * removed / self.n_start, 4) if self.n_start else 0.0,
            "rows_remaining": n_after,
            "detail": detail,
        })
        self.n_current = n_after

    def note(self, key: str, value) -> None:
        self.stats[key] = value

    def to_dict(self) -> dict:
        return {
            "dataset": self.dataset,
            "rows_in": self.n_start,
            "rows_out": self.n_current,
            "retained_pct": round(100 * self.n_current / self.n_start, 2) if self.n_start else 0.0,
            "elapsed_sec": round(time.time() - self.t0, 1),
            "steps": self.steps,
            "stats": self.stats,
        }

    def save(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(self.to_dict(), indent=2, ensure_ascii=False,
                                   default=str), encoding="utf-8")

    def print_table(self) -> None:
        print(f"\n--- cleaning audit: {self.dataset} ---")
        print(f"{'step':<5}{'reason':<38}{'removed':>10}{'%orig':>9}{'left':>12}")
        for s in self.steps:
            print(f"{s['step']:<5}{s['reason']:<38}{s['removed']:>10,}"
                  f"{s['pct_of_original']:>8.2f}%{s['rows_remaining']:>12,}")
        print(f"{'':5}{'TOTAL RETAINED':<38}{'':>10}"
              f"{100 * self.n_current / self.n_start if self.n_start else 0:>8.2f}%"
              f"{self.n_current:>12,}")

## Part 3 — `01_clean_primary`: clean the SARC balanced Reddit corpus

Uses everything defined in Parts 1 and 2 directly (no `tc.` prefix needed,
since it's all in this same notebook namespace) — this only processes
`train-balanced-sarcasm.csv` (set in `SRC`); it does not touch a separate
test/held-out file if one exists.

### Configuration

In [9]:
SRC = "../../data/raw/primary/reddit_sarc/train-balanced-sarcasm.csv"
NAME = "reddit_sarc_balanced"

DTYPES = {
    "label": "int8", "comment": "string", "author": "string",
    "subreddit": "string", "score": "float32", "ups": "float32",
    "downs": "float32", "date": "string", "parent_comment": "string",
}

MIN_WORDS = 2
MAX_WORDS = 300          # 99.99th pct is ~130; beyond this it is copypasta
MAX_CHARS = 2000

# --- CLI-equivalent parameters ---
SAMPLE_SIZE: int | None = None   # e.g. 200_000 for a sampled run; None = full corpus
SEED = 3244
USE_LANGDETECT = True            # set False for --no-langdetect
SAMPLE_OUT = 5000                # rows written to the human-inspectable CSV sample

### Step 1 — load + type-coerce

In [10]:
def load(sample: int | None, seed: int) -> tuple[pd.DataFrame, int]:
    print(f"[load] {SRC}")
    df = pd.read_csv(SRC, dtype=DTYPES, parse_dates=["created_utc"],
                     on_bad_lines="warn", low_memory=False)
    n_full = len(df)
    print(f"       {n_full:,} rows x {df.shape[1]} cols")
    if sample and sample < n_full:
        # Stratify on the label so the class balance of the sample matches
        # the corpus; the corpus is balanced by construction, so this just
        # guarantees we don't introduce skew by luck.
        parts = [g.sample(n=min(len(g), int(round(sample * len(g) / n_full))),
                          random_state=seed)
                 for _, g in df.groupby("label", sort=True)]
        df = pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
        print(f"[load] stratified sample -> {len(df):,} rows")
    return df, n_full

In [11]:
df, n_full = load(SAMPLE_SIZE, SEED)
df.head()

[load] ../../data/raw/primary/reddit_sarc/train-balanced-sarcasm.csv
       1,010,826 rows x 10 cols


,label,comment,author,subreddit,score,ups,downs,date,created_utc,parent_comment
0,0,NC and NH.,Trumpbart,politics,2.0,-1.0,-1.0,2016-10,2016-10-16 23:55:23,"Yeah, I get that argument. At this point, I'd ..."
1,0,You do know west teams play against west teams...,Shbshb906,nba,-4.0,-1.0,-1.0,2016-11,2016-11-01 00:24:10,The blazers and Mavericks (The wests 5 and 6 s...
2,0,"They were underdogs earlier today, but since G...",Creepeth,nfl,3.0,3.0,0.0,2016-09,2016-09-22 21:45:37,They're favored to win.
3,0,"This meme isn't funny none of the ""new york ni...",icebrotha,BlackPeopleTwitter,-8.0,-1.0,-1.0,2016-10,2016-10-18 21:03:47,deadass don't kill my buzz
4,0,I could use one of those tools.,cush2push,MaddenUltimateTeam,6.0,-1.0,-1.0,2016-12,2016-12-30 17:00:13,Yep can confirm I saw the tool they use for th...


### Steps 2–7 — cleaning

Drop unusable comments, normalise text, strip label leakage, enforce length bounds, filter to English, and de-duplicate. Calls the Part 1 functions directly (`tc.clean_series` etc. are now just `clean_series`).

In [12]:
def clean(df: pd.DataFrame, log: CleaningLog, use_langdetect: bool) -> pd.DataFrame:
    # -- 2. unusable comments ---------------------------------------------
    bad = is_deleted(df["comment"]) | df["comment"].isna()
    df = df[~bad]
    log.drop("empty / [deleted] / [removed] comment", len(df))

    # -- 3-4. text pipeline (normalise, strip markup, de-leak) ------------
    print("[clean] normalising comment text ...")
    ctext = clean_series(df["comment"], keep_subreddit=True)
    log.note("leak_slash_s_in_comment", int(ctext["had_slash_s"].sum()))
    log.note("leak_label_hashtag_in_comment", int(ctext["had_label_hashtag"].sum()))
    log.note("leak_explicit_word_in_comment", int(ctext["had_explicit_sarcasm_word"].sum()))

    # Leak rate broken down by class tells us whether the marker really is
    # the label in disguise.
    leak_by_label = (ctext["had_slash_s"] | ctext["had_explicit_sarcasm_word"]) \
        .groupby(df["label"]).mean().round(6).to_dict()
    log.note("leak_rate_by_label", {int(k): float(v) for k, v in leak_by_label.items()})

    print("[clean] normalising parent_comment text ...")
    ptext = clean_series(df["parent_comment"].fillna(""), keep_subreddit=True)

    df = df.assign(
        comment_clean=ctext["text_clean"].values,
        parent_clean=ptext["text_clean"].values,
        **{c: ctext[c].values for c in ctext.columns
           if c not in ("text_clean", "text_norm")},
    )
    df["parent_n_words"] = ptext["n_words"].values
    df["parent_n_chars"] = ptext["n_chars"].values
    df["parent_is_empty"] = (ptext["n_words"].values == 0)

    # Text can become empty once the marker and markup are gone — e.g. a
    # comment that was literally just "/s".
    empty_after = df["comment_clean"].str.strip().eq("")
    log.note("became_empty_after_cleaning", int(empty_after.sum()))
    df = df[~empty_after]
    log.drop("empty after markup/leak removal", len(df))

    # -- 5. length bounds --------------------------------------------------
    too_short = df["n_words"] < MIN_WORDS
    df = df[~too_short]
    log.drop(f"fewer than {MIN_WORDS} words", len(df))

    too_long = (df["n_words"] > MAX_WORDS) | (df["n_chars"] > MAX_CHARS)
    df = df[~too_long]
    log.drop(f"more than {MAX_WORDS} words / {MAX_CHARS} chars", len(df))

    # -- 6. language -------------------------------------------------------
    print("[clean] language filtering ...")
    df = add_language_flags(df, "comment_clean", use_langdetect=use_langdetect)
    log.note("lang_method_counts", df["lang_method"].value_counts().to_dict())
    df = df[df["is_english"]]
    log.drop("non-English (heuristic + langdetect)", len(df))

    # -- 7. de-duplication -------------------------------------------------
    print("[clean] de-duplicating ...")
    df["_key"] = dedup_key(df["comment_clean"])

    before = len(df)
    df = df.drop_duplicates(subset=["_key", "label", "parent_clean"], keep="first")
    log.drop("exact dup (comment+label+parent)", len(df),
             "same comment under the same parent with the same label")

    # A comment whose normalised form appears with BOTH labels is
    # irreducibly ambiguous — the identical string is sarcastic in one thread
    # and sincere in another. Keeping both sides only teaches the model noise.
    conflict = df.groupby("_key")["label"].transform("nunique") > 1
    log.note("label_conflicting_texts", int(df.loc[conflict, "_key"].nunique()))
    log.note("label_conflicting_rows", int(conflict.sum()))
    df = df[~conflict]
    log.drop("same text carries both labels", len(df))

    df = df.drop_duplicates(subset=["_key"], keep="first")
    log.drop("near-dup (normalised text, any parent)", len(df),
             f"{before - len(df):,} total duplicate rows removed")
    df = df.drop(columns=["_key"])

    return df

### Step 8 — feature engineering

Context + temporal + score features used by the EDA and the models.

In [13]:
def engineer(df: pd.DataFrame) -> pd.DataFrame:
    """Context + temporal + score features used by the EDA and the models."""
    print("[feat] context & temporal features ...")
    df["parent_jaccard"] = [
        jaccard(a, b) for a, b in zip(df["comment_clean"], df["parent_clean"])
    ]
    df["len_ratio_to_parent"] = (
        df["n_words"] / df["parent_n_words"].replace(0, np.nan)
    ).astype("float32")

    ts = pd.to_datetime(df["created_utc"], errors="coerce")
    df["year"] = ts.dt.year.astype("Int16")
    df["month"] = ts.dt.month.astype("Int8")
    df["hour_utc"] = ts.dt.hour.astype("Int8")
    df["weekday"] = ts.dt.dayofweek.astype("Int8")

    # `ups`/`downs` are -1 for most of the corpus (Reddit stopped exposing
    # them); `score` is the field that actually carries information.
    df["score"] = df["score"].fillna(0).astype("float32")
    df["ups_is_missing"] = df["ups"].fillna(-1).eq(-1)
    df["downs_is_missing"] = df["downs"].fillna(-1).eq(-1)
    df["is_controversial"] = df["score"].between(-1, 1)

    df["subreddit"] = df["subreddit"].fillna("<unknown>").astype("string")
    df["author"] = df["author"].fillna("<unknown>").astype("string")

    # Crude sentiment-incongruity proxy: positive surface words in a comment
    # that got downvoted. Replaced by the lexicon features in step 04.
    df["has_positive_interjection"] = df["starts_with_interjection"] & (df["n_words"] > 2)
    return df

### Column ordering for the final output

In [14]:
ORDERED_COLS = [
    "label", "comment_clean", "parent_clean", "subreddit", "author",
    "score", "ups", "downs", "ups_is_missing", "downs_is_missing",
    "is_controversial", "date", "created_utc", "year", "month", "hour_utc",
    "weekday", "n_chars", "n_words", "n_sentences", "avg_word_len",
    "n_exclam", "n_question", "n_ellipsis", "n_repeat_punct",
    "n_allcaps_words", "allcaps_ratio", "n_emoji", "n_elongation",
    "n_scare_quotes", "n_urls", "n_mentions", "n_hashtags",
    "n_interjections", "starts_with_interjection", "has_positive_interjection",
    "parent_n_words", "parent_n_chars", "parent_is_empty", "parent_jaccard",
    "len_ratio_to_parent", "had_slash_s", "had_label_hashtag",
    "had_explicit_sarcasm_word", "en_score", "lang_method",
]

### Run the pipeline

In [15]:
#ensure_dirs()

log = CleaningLog(NAME, len(df))
log.note("source_file", str(SRC))
log.note("rows_in_source_file", n_full)
log.note("sampled", bool(SAMPLE_SIZE))

df = clean(df, log, use_langdetect=USE_LANGDETECT)
df = engineer(df)

df = df[[c for c in ORDERED_COLS if c in df.columns]].reset_index(drop=True)

log.note("final_label_balance", df["label"].value_counts().to_dict())
log.note("n_subreddits", int(df["subreddit"].nunique()))
log.note("n_authors", int(df["author"].nunique()))
log.note("date_range", [str(df["created_utc"].min()), str(df["created_utc"].max())])

df.head()

[clean] normalising comment text ...
[clean] normalising parent_comment text ...
[clean] language filtering ...
[clean] de-duplicating ...
[feat] context & temporal features ...


,label,comment_clean,parent_clean,subreddit,author,score,ups,downs,ups_is_missing,downs_is_missing,...,parent_n_words,parent_n_chars,parent_is_empty,parent_jaccard,len_ratio_to_parent,had_slash_s,had_label_hashtag,had_explicit_sarcasm_word,en_score,lang_method
0,0,NC and NH.,"Yeah, I get that argument. At this point, I'd ...",politics,Trumpbart,2.0,-1.0,-1.0,True,True,...,17,80,False,0.052632,0.176471,False,False,False,1.000000,heuristic-high
1,0,You do know west teams play against west teams...,The blazers and Mavericks (The wests 5 and 6 s...,nba,Shbshb906,-4.0,-1.0,-1.0,True,True,...,25,134,False,0.032258,0.560000,False,False,False,0.900000,heuristic-high
2,0,"They were underdogs earlier today, but since G...",They're favored to win.,nfl,Creepeth,3.0,3.0,0.0,False,False,...,4,23,False,0.047619,4.500000,False,False,False,0.990278,heuristic-high
3,0,"This meme isn't funny none of the ""new york ni...",deadass don't kill my buzz,BlackPeopleTwitter,icebrotha,-8.0,-1.0,-1.0,True,True,...,5,26,False,0.000000,2.400000,False,False,False,1.000000,heuristic-high
4,0,I could use one of those tools.,Yep can confirm I saw the tool they use for th...,MaddenUltimateTeam,cush2push,6.0,-1.0,-1.0,True,True,...,19,85,False,0.083333,0.368421,False,False,False,1.000000,heuristic-high


### Save outputs

Writes the cleaned parquet, a stratified CSV sample for manual inspection, and the JSON cleaning-audit report.

In [17]:
suffix = f"_sample{SAMPLE_SIZE}" if SAMPLE_SIZE else "_full"
PROCESSED = Path("../../data/processed")
NAME = "reddit_sarc_balanced"

out_parquet = PROCESSED / f"{NAME}{suffix}.parquet"
df.to_parquet(out_parquet, index=False)
print(f"\n[save] {out_parquet}  ({out_parquet.stat().st_size / 1e6:.1f} MB)")

#csv_sample = SAMPLES / f"{NAME}{suffix}_head{SAMPLE_OUT}.csv"
#df.sample(n=min(SAMPLE_OUT, len(df)), random_state=SEED) \
#  .sort_index().to_csv(csv_sample, index=False, encoding="utf-8")
#print(f"[save] {csv_sample}")

#log.print_table()
#rep = CLEAN_REPORTS / f"{NAME}{suffix}_cleaning_report.json"
#log.save(rep)
#print(f"[save] {rep}")


[save] ..\..\data\processed\reddit_sarc_balanced_full.parquet  (163.8 MB)
